In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/amazon-ml-text-sbert-embeddings/train_features_v5_sbert_embeddings_384.npy
/kaggle/input/amazon-ml-text-sbert-embeddings/test_features_v5_sbert_embeddings_384.npy
/kaggle/input/amazon-ml-embeddings/test_vit_embeddings.npy
/kaggle/input/amazon-ml-embeddings/train_features_competitive.csv
/kaggle/input/amazon-ml-embeddings/train_features_v5.csv
/kaggle/input/amazon-ml-embeddings/test_features_competitive.csv
/kaggle/input/amazon-ml-embeddings/train_cnn_embeddings.npy
/kaggle/input/amazon-ml-embeddings/test_cnn_embeddings.npy
/kaggle/input/amazon-ml-embeddings/train_cleaned.csv
/kaggle/input/amazon-ml-embeddings/train_features_v3.csv
/kaggle/input/amazon-ml-embeddings/test_features_v5.csv
/kaggle/input/amazon-ml-embeddings/train_features_v4.csv
/kaggle/input/amazon-ml-embeddings/train_vit_embeddings.npy
/kaggle/input/amazon-ml-data-new/new_merged_output_test_2.csv
/kaggle/input/amazon-ml-data-new/new_merged_output_train_2.csv


In [2]:
!nvidia-smi

Mon Oct 13 12:41:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             34W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
pip install xgboost==2.1.1 --upgrade

In [3]:
import time
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import KFold
from scipy.sparse import hstack, csr_matrix

In [4]:
# --- Configuration ---
TABULAR_TRAIN_PATH = '/kaggle/input/amazon-ml-data-new/new_merged_output_train_2.csv'
IMAGE_EMBEDDINGS_PATH = '/kaggle/input/amazon-ml-embeddings/train_vit_embeddings.npy'
TEXT_FEATURES_PATH = '/kaggle/input/amazon-ml-text-sbert-embeddings/train_features_v5_sbert_embeddings_384.npy'
N_TRIALS = 1 # 50 
N_SPLITS_CV = 3 
TARGET_COL = "price"

In [5]:
# =========================================================
# --- Helper Functions ---
# =========================================================
def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / (denominator + 1e-8)) * 100

def smoothed_target_encoding(train_series, target, smoothing=10):
    global_mean = target.mean()
    stats = target.groupby(train_series).agg(['count', 'mean'])
    smoothed_mean = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    encoded_map = smoothed_mean.to_dict()
    return train_series.map(encoded_map).fillna(global_mean)

# =========================================================
# --- Load Data ---
# =========================================================
print("📥 Loading tabular data...")
df = pd.read_csv(TABULAR_TRAIN_PATH)
print(f"✅ Tabular shape: {df.shape}")

print("📥 Loading image embeddings...")
image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

print("📥 Loading text embeddings...")
text_features = np.load(TEXT_FEATURES_PATH)

# Target variable
y = np.log1p(df[TARGET_COL])

# =========================================================
# --- Encode Categorical Columns ---
# =========================================================
cat_cols = ['unit_field_parsed', 'conversion_method', 'density_key', 'product_type', 'Brand_Name']
for c in cat_cols:
    df[c] = df[c].astype(str).fillna("unknown")

# Smoothed target encoding for Brand_Name
df['brand_enc'] = smoothed_target_encoding(df['Brand_Name'], y)

# Label encode other categoricals
le = LabelEncoder()
for c in cat_cols:
    df[c + "_enc"] = le.fit_transform(df[c])

# =========================================================
# --- Numerical Columns ---
# =========================================================
numerical_cols = [
    'value_field_parsed', 'pack_from_text', 'grams_total', 'ml_total',
    'per_item_g', 'per_item_ml', 'confidence', 'density_used',
    'density_confidence', 'Pack_Quantity',
    'max_num_in_name', 'min_num_in_name', 'avg_num_in_name', 'num_count_in_name',
    'brand_enc'
]

scaler = MinMaxScaler()
X_num = scaler.fit_transform(df[numerical_cols].fillna(0))

cat_features = df[[c + "_enc" for c in cat_cols]].values

# =========================================================
# --- Combine All Features with hstack ---
# =========================================================
X = hstack([
    csr_matrix(X_num),
    csr_matrix(cat_features),
    csr_matrix(image_embeddings),
    csr_matrix(text_features)
]).tocsr()

print(f"✅ Combined feature matrix shape: {X.shape}")

# =========================================================
# --- Optuna Objective Function ---
# =========================================================
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "gpu_id": 0,
        "verbosity": 0,
        "max_depth": trial.suggest_int("max_depth", 5, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 5e-4, 10.0, log=True),
        "n_estimators": 5000,
        "random_state": 42
    }

    kf = KFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=42)
    smape_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Convert training/validation data to DMatrix
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)

        model = xgb.train(
                params,
                dtrain,
                num_boost_round=4000,
                evals=[(dval, "validation")],
                early_stopping_rounds=150,
                verbose_eval=False  # set to True for per-round output
            )

        preds = model.predict(dval)

        # preds = model.predict(X_val)
        smape_scores.append(smape(y_val, preds))

    return np.mean(smape_scores)



📥 Loading tabular data...
✅ Tabular shape: (75000, 21)
📥 Loading image embeddings...
📥 Loading text embeddings...
✅ Combined feature matrix shape: (75000, 1940)


In [ ]:
# =========================================================
# --- Run Optuna Study ---
# =========================================================
optuna.logging.set_verbosity(optuna.logging.INFO)
start_time = time.time()

study = optuna.create_study(direction="minimize", study_name="amazon_gpu_optuna")

for i in range(N_TRIALS):
    trial_start = time.time()
    trial = study.ask()
    value = objective(trial)
    study.tell(trial, value)

    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (N_TRIALS - (i + 1))
    print(f"[Trial {i+1}/{N_TRIALS}] SMAPE = {value:.4f} | ETA ≈ {remaining/60:.1f} min")

[I 2025-10-13 12:41:54,611] A new study created in memory with name: amazon_gpu_optuna
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:41:57] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)


In [ ]:
model.save_model("xgboost_new_1_json.json")

In [ ]:
import pickle

# For XGBRegressor
pickle.dump(model, open("xgboost_new_l_pkl.pkl", "wb"))

In [ ]:
# =========================================================
# --- Results ---
# =========================================================
print("\n" + "="*70)
print("🎯 OPTIMIZATION COMPLETE (GPU + hstack)")
print(f"Best SMAPE: {study.best_value:.4f}")
print("Best Hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("="*70)